# paper execution objective loss gate v1

验证 `objective.loss.v1` 从 dashboard export 进入 core readiness、API intent、schedule paper execution payload 和通知。

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / 'research' / 'paper_execution_objective_loss_gate_v1'
CARD.mkdir(parents=True, exist_ok=True)
print(f'ROOT={ROOT}')
print(f'CARD={CARD}')

ROOT=/Users/rziiiii/dev/workspace
CARD=/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1


In [2]:
import pandas as pd

from src.api import build_target_position_intent_from_dashboard
from src.core import (
    DatasetSnapshot,
    build_architecture_extension_decision,
    publish_production_manifest,
    write_architecture_extension_decision,
)
from src.data import build_canonical_snapshot, publish_canonical_snapshot
from src.schedule import PaperExecutionPipelineConfig, run_paper_execution_once

In [3]:
raw = pd.DataFrame([
    {'date': '2026-01-02', 'instrument': 'SOXL', 'close': 10.0, 'volume': 100},
    {'date': '2026-01-05', 'instrument': 'SOXL', 'close': 12.0, 'volume': 120},
])
calendar = pd.DatetimeIndex(pd.to_datetime(['2026-01-02', '2026-01-05']), name='date')
snapshot = build_canonical_snapshot(
    raw,
    name='paper_execution_objective_loss_gate_soxl_daily',
    universe=('SOXL',),
    calendar=calendar,
    calendar_path='data/qlib/soxl/calendars/day.txt',
    source_paths=('provider:notebook_unit',),
    start='2026-01-01',
    end='2026-01-05',
    as_of='2026-01-05',
)
registry_manifest = publish_canonical_snapshot(
    snapshot,
    output_dir=CARD / 'data' / 'snapshots' / 'unit_soxl_daily' / '2026-01-05',
    dataset_key='daily.unit_soxl_objective_loss_gate',
    registry_path=CARD / 'data' / 'data_registry.json',
    published_at='2026-06-19T00:00:00Z',
)
registry_path = CARD / 'data' / 'data_registry.json'
registry_manifest['manifest_hash'][:12]

'003157b0fb60'

In [4]:
def dashboard_payload(*, reward_surface: str, rank_group_count: int, artifact_id: str) -> dict:
    return {
        'schema_version': 'dashboard.model_matrix.v1',
        'artifact_id': artifact_id,
        'artifact_type': 'dashboard_export',
        'manifest_source': {
            'kind': 'analysis_manifest',
            'path': 'matrix_card/analysis_manifest.json',
            'manifest_hash': 'c' * 64,
            'producer': 'src.analysis.exports',
        },
        'top_models': [
            {
                'rank': 1,
                'model': 'ridge_mse__all_features',
                'base_model': 'ridge_mse',
                'feature_set': 'all_features',
                'avg_position': 0.5,
            }
        ],
        'objective': {
            'losses': {
                'schema_version': 'objective.loss.v1',
                'metric_source_schema_version': 'objective.evaluation.v1',
                'optimization_direction': 'minimize',
                'reward_surface': reward_surface,
                'row_count': 78,
                'mse_loss': 0.0274,
                'classification_loss': 0.4359,
                'ic_loss': 0.9774,
                'ranking_loss': 1.0 if rank_group_count == 0 else 0.2,
                'rank_group_count': rank_group_count,
            }
        },
    }

valid_dashboard_path = CARD / 'dashboard_export.json'
valid_dashboard_path.write_text(
    json.dumps(dashboard_payload(reward_surface='retired', rank_group_count=0, artifact_id='objective_loss_gate_valid'), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
invalid_dashboard_path = CARD / 'dashboard_export_invalid_reward.json'
invalid_dashboard_path.write_text(
    json.dumps(dashboard_payload(reward_surface='active_rl_reward', rank_group_count=12, artifact_id='objective_loss_gate_invalid_reward'), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
valid_dashboard_path

PosixPath('/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1/dashboard_export.json')

In [5]:
events: list[str] = []
valid_config = PaperExecutionPipelineConfig(
    name='paper_execution',
    dashboard_export_path=valid_dashboard_path,
    data_registry_path=registry_path,
    dataset_key='daily.unit_soxl_objective_loss_gate',
    instrument='SOXL',
    intent_ledger_path=CARD / 'intent_ledger.json',
    paper_execution_result_path=CARD / 'paper_execution_result.json',
    initial_cash=1000.0,
    notify_details=True,
)
valid_result = run_paper_execution_once(valid_config, lock_dir=CARD / 'locks', notify=events.append)
valid_ledger = json.loads(valid_config.intent_ledger_path.read_text(encoding='utf-8'))

invalid_config = PaperExecutionPipelineConfig(
    name='paper_execution_invalid_reward',
    dashboard_export_path=invalid_dashboard_path,
    data_registry_path=registry_path,
    dataset_key='daily.unit_soxl_objective_loss_gate',
    instrument='SOXL',
    intent_ledger_path=CARD / 'intent_ledger_invalid_reward.json',
    paper_execution_result_path=CARD / 'paper_execution_result_invalid_reward.json',
    initial_cash=1000.0,
    min_position=0.1,
    max_position=0.75,
    notify_details=True,
)
invalid_result = run_paper_execution_once(invalid_config, lock_dir=CARD / 'locks', notify=events.append)
invalid_ledger = json.loads(invalid_config.intent_ledger_path.read_text(encoding='utf-8'))

assert valid_result.status == 'success'
assert valid_result.payload['execution_gate'] == 'paper_review'
assert valid_result.payload['readiness_overview_status'] == 'review'
assert valid_result.payload['readiness_overview_review_gates'] == 'objective_loss'
assert valid_result.payload['objective_loss_status'] == 'review'
assert valid_result.payload['objective_loss_reward_surface'] == 'retired'
assert valid_result.payload['objective_loss_top_component'] == 'ranking_loss'
assert valid_ledger['intent']['metadata']['objective_loss_status'] == 'review'

assert invalid_result.status == 'success'
assert invalid_result.payload['execution_gate'] == 'blocked_to_min_position'
assert invalid_result.payload['target_position'] == 0.1
assert invalid_result.payload['readiness_overview_status'] == 'blocked'
assert invalid_result.payload['readiness_overview_block_gates'] == 'objective_loss'
assert invalid_result.payload['objective_loss_status'] == 'block'
assert invalid_result.payload['objective_loss_reward_surface'] == 'active_rl_reward'
assert invalid_ledger['intent']['metadata']['objective_loss_status'] == 'block'

events

['paper_execution:success | mode=paper | gate=paper_review | model=ridge_mse__all_features | target=0.5 | dashboard_manifest=analysis_manifest/complete | quote=SOXL@2026-01-05/12 | shares=41 | ledger=/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1/intent_ledger.json | objective_loss=objective.loss.v1/review top=ranking_loss:1 reward=retired',
 'paper_execution_invalid_reward:success | mode=paper | gate=blocked_to_min_position | model=ridge_mse__all_features | target=0.1 | dashboard_manifest=analysis_manifest/complete | quote=SOXL@2026-01-05/12 | shares=8 | ledger=/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1/intent_ledger_invalid_reward.json | objective_loss=objective.loss.v1/block top=ic_loss:0.9774 reward=active_rl_reward']

In [6]:
decision = build_architecture_extension_decision(
    decision_id='paper-execution-objective-loss-gate-v1',
    target_surface='src/core,src/api,src/schedule',
    change_type='execution_readiness_gate',
    reason='需要把 objective.loss.v1 从 dashboard 证据扩展到 paper 执行前 gate，防止 reward surface 或损失契约异常时继续按原仓位执行。',
    impact_scope=(
        'src/core/contracts.py',
        'src/api/intents.py',
        'src/schedule/pipelines.py',
        'research/paper_execution_objective_loss_gate_v1',
    ),
    evidence_notebook='research/paper_execution_objective_loss_gate_v1/paper_execution_objective_loss_gate_v1.ipynb',
    verification_commands=(
        'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests/test_platform_contracts.py tests/test_api_paper_intents.py tests/test_schedule_paper_execution_pipeline.py -q',
        'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests -q',
    ),
    alternatives_considered=(
        '只在 dashboard 页面展示 objective loss，不进入执行前 gate',
        '在 schedule 通知里拼接字符串但不写入 intent metadata',
    ),
    constraints=(
        '不新增 RL/reward 训练面，reward_surface 必须保持 retired',
        'paper execution v1 只允许 paper/sim，不触发真实 broker 下单',
    ),
    created_at='2026-06-19T00:00:00Z',
)
decision_result = write_architecture_extension_decision(
    output_dir=CARD,
    decision=decision,
    registry_path=CARD / 'architecture_extension_registry.json',
)
decision_result

{'schema_version': 'core.architecture_extension_write_result.v1',
 'decision_path': '/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1/paper-execution-objective-loss-gate-v1.json',
 'registry_path': '/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1/architecture_extension_registry.json',
 'review_path': '/Users/rziiiii/dev/workspace/research/paper_execution_objective_loss_gate_v1/architecture_extension_review.md',
 'decision_hash': '84a0791af69fc2686ba2441eaabaed23e5bbc67f06ce7a2fa491321775d6dbd7',
 'registry_hash': '1126bae7099faf07ba8a8da3ade7a2b54b68912a3a6ccd82323d23d82d6a82af'}

In [7]:
summary = {
    'schema_version': 'research.paper_execution_objective_loss_gate.v1',
    'status': 'completed',
    'valid_reward_surface': {
        'job_status': valid_result.status,
        'execution_gate': valid_result.payload['execution_gate'],
        'target_position': valid_result.payload['target_position'],
        'readiness_overview_status': valid_result.payload['readiness_overview_status'],
        'readiness_overview_review_gates': valid_result.payload['readiness_overview_review_gates'],
        'objective_loss_schema_version': valid_result.payload['objective_loss_schema_version'],
        'objective_loss_status': valid_result.payload['objective_loss_status'],
        'objective_loss_reward_surface': valid_result.payload['objective_loss_reward_surface'],
        'objective_loss_top_component': valid_result.payload['objective_loss_top_component'],
        'objective_loss_top_value': valid_result.payload['objective_loss_top_value'],
    },
    'invalid_reward_surface': {
        'job_status': invalid_result.status,
        'execution_gate': invalid_result.payload['execution_gate'],
        'target_position': invalid_result.payload['target_position'],
        'readiness_overview_status': invalid_result.payload['readiness_overview_status'],
        'readiness_overview_block_gates': invalid_result.payload['readiness_overview_block_gates'],
        'objective_loss_schema_version': invalid_result.payload['objective_loss_schema_version'],
        'objective_loss_status': invalid_result.payload['objective_loss_status'],
        'objective_loss_reward_surface': invalid_result.payload['objective_loss_reward_surface'],
        'objective_loss_top_component': invalid_result.payload['objective_loss_top_component'],
        'objective_loss_top_value': invalid_result.payload['objective_loss_top_value'],
    },
    'notification_events': events,
    'decision_result': decision_result,
    'conclusion': 'objective.loss.v1 已进入 paper execution readiness 和 intent metadata；retired reward surface 进入 review，active_rl_reward 被 block 到最小仓位。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(CARD / 'notification_events.json').write_text(json.dumps({'events': events}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary

{'schema_version': 'research.paper_execution_objective_loss_gate.v1',
 'status': 'completed',
 'valid_reward_surface': {'job_status': 'success',
  'execution_gate': 'paper_review',
  'target_position': 0.5,
  'readiness_overview_status': 'review',
  'readiness_overview_review_gates': 'objective_loss',
  'objective_loss_schema_version': 'objective.loss.v1',
  'objective_loss_status': 'review',
  'objective_loss_reward_surface': 'retired',
  'objective_loss_top_component': 'ranking_loss',
  'objective_loss_top_value': 1.0},
 'invalid_reward_surface': {'job_status': 'success',
  'execution_gate': 'blocked_to_min_position',
  'target_position': 0.1,
  'readiness_overview_status': 'blocked',
  'readiness_overview_block_gates': 'objective_loss',
  'objective_loss_schema_version': 'objective.loss.v1',
  'objective_loss_status': 'block',
  'objective_loss_reward_surface': 'active_rl_reward',
  'objective_loss_top_component': 'ic_loss',
  'objective_loss_top_value': 0.9774},
 'notification_even

## 中文结论

- 做了什么：把 `objective.loss.v1` 从 dashboard export 接入 `src/core` readiness gate、`src/api` paper intent metadata、`src/schedule` paper execution payload 和通知。
- 核心指标：`reward_surface=retired` 且 `ranking_loss=1.0/rank_group_count=0` 时，paper execution 保持 `paper_review`，readiness 为 `review`，通知包含 objective loss 摘要；`reward_surface=active_rl_reward` 时，readiness 为 `blocked`，执行 gate 为 `blocked_to_min_position`，目标仓位降到 0.1。
- regime 差异：本实验不比较市场 regime，只验证执行前契约 gate；不同区间/regime 的效果差异仍由 `src/analysis` notebook 负责。
- 是否晋升：晋升为框架能力，不代表策略晋升；v1 仍只允许 paper/sim。
- 风险：loss 阈值目前只做契约级 gate，具体模型优劣仍需 backtest、analysis、promotion 联合判断。
- 下一步：把 production health 读取 paper execution objective loss 摘要，形成生产方案健康面板的一致证据。
